# 05. A/B 테스트 설계 문서 — 리텐션 캠페인

⚠️ **이 노트북은 실제 실험이 아니라 설계 문서다.** 실제로 캠페인(이메일/푸시 등)을 발송하거나
그 결과를 분석하지 않는다. 위험 세그먼트 정의, 표본크기·기간·검정 방법을 미리 정해두는
"제안서" 성격의 산출물이다. 캠페인의 구체적 채널·인센티브 내용은 이 문서의 범위 밖이며,
실행 시점에 별도로 정한다.

## 목표
04에서 확정한 이탈 정의(N=30일 무활동)를 바탕으로, 아직 30일 임계값을 넘기지 않았지만
임박한 유저("위험 세그먼트")를 대상으로 한 리텐션 캠페인을 어떻게 설계·검증할지 정한다.

## 1. 위험 세그먼트 정의

**합의된 방향**: 이미 `churned` 확정된 유저는 캠페인 타겟으로는 늦음(윈백 대상이지 리텐션
방지 대상이 아님). 아직 30일 임계값을 안 넘겼지만 임박한 유저(recency 15~29일)를
"위험 세그먼트"로 정의한다.

⚠️ **주의 — 기준 시점 문제**: `user_features.parquet`의 `recency_days`는 데이터 마지막 날
(`dataset_end` = 2020-04-30) 기준으로 계산되어 있다. 이 컬럼으로 위험 세그먼트를 뽑으면
순환논리에 빠진다 — recency 15~29일(=마지막 활동이 4/1~4/15)이라는 것 자체가 이미
`reference_date`(3/31) 기준 재방문(retained)에 해당하는 조건과 겹쳐서, 이 세그먼트에는
`churned`가 단 한 명도 나오지 않는다(직접 확인함). 즉 "아직 결과를 모르는 위험군"이 아니라
"이미 재방문에 성공한 사람들"만 걸러지는 셈이라 baseline 재방문율 계산에 쓸 수 없다.

**해결**: "오늘"을 데이터 마지막 날이 아니라 04에서 이미 쓰고 있는 `reference_date`
(2020-03-31)로 앞당겨서 recency를 다시 계산한다. 이 시점 기준이면 위험 세그먼트는
"아직 30일 관찰 창이 안 끝나 결과가 불확실했던" 진짜 위험군이 되고, 그 이후 실제
재방문 여부(`churn_labels`의 status)로 baseline 재방문율을 구할 수 있다.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import config
from src import load

# reference_date(3/31) 이전 각 유저의 마지막 활동일을 다시 구한다.
# 4월 데이터는 이 계산에 필요 없으므로 2019-10~2020-03, 6개월치만 로드한다
# (user_id, event_time 2개 컬럼만 + config.DTYPES 지정 — 04와 동일한 메모리 전략).
month_labels = ["2019-10", "2019-11", "2019-12", "2020-01", "2020-02", "2020-03"]
paths = config.RAW_PATHS[:6]

daily_frames = []
for label, path in zip(month_labels, paths):
    df = load.load_month(path, usecols=["user_id", "event_time"])
    df["date"] = df["event_time"].dt.normalize()
    daily = df[["user_id", "date"]].drop_duplicates()
    del df
    daily_frames.append(daily)
    print(f"{label}: 유저-활동일 조합 {len(daily):,}건")

user_activity_days_pre = pd.concat(daily_frames, ignore_index=True)
del daily_frames

last_date_before_ref = user_activity_days_pre.groupby("user_id")["date"].max()
last_date_before_ref.name = "last_date_before_ref"

reference_date = pd.Timestamp("2020-03-31", tz=getattr(last_date_before_ref.dtype, "tz", None))
recency_at_ref = (reference_date - last_date_before_ref).dt.days
recency_at_ref.name = "recency_at_ref"

recency_at_ref_df = recency_at_ref.reset_index()
recency_at_ref_df.to_parquet(config.PROC_DIR / "recency_at_reference.parquet", index=False)

print("saved:", recency_at_ref_df.shape)
recency_at_ref_df["recency_at_ref"].describe()

2019-10: 유저-활동일 조합 6,473,723건


2019-11: 유저-활동일 조합 8,621,421건


2019-12: 유저-활동일 조합 9,984,617건


2020-01: 유저-활동일 조합 9,039,523건


2020-02: 유저-활동일 조합 8,526,097건


2020-03: 유저-활동일 조합 8,325,825건


saved: (13529831, 2)


count    1.352983e+07
mean     6.903427e+01
std      4.973823e+01
min      0.000000e+00
25%      2.500000e+01
50%      6.000000e+01
75%      1.040000e+02
max      1.820000e+02
Name: recency_at_ref, dtype: float64

In [2]:
churn_labels = pd.read_parquet(config.PROC_DIR / "churn_labels.parquet")[["user_id", "status"]].set_index("user_id")
user_features = pd.read_parquet(config.PROC_DIR / "user_features.parquet")[["user_id", "is_buyer"]].set_index("user_id")

df = recency_at_ref.to_frame().join(churn_labels).join(user_features)
print("reference_date 이전 평가 대상:", len(df))
print(df["status"].value_counts())

risk = df[(df["recency_at_ref"] >= 15) & (df["recency_at_ref"] <= 29)]
print("\n위험 세그먼트 크기:", len(risk))
print(risk["status"].value_counts())

baseline_overall = (risk["status"] == "retained").mean()
baseline_by_segment = risk.groupby("is_buyer")["status"].apply(lambda s: (s == "retained").mean())
segment_size = risk.groupby("is_buyer").size()

print(f"\nbaseline 재방문율 (전체): {baseline_overall:.2%}")
print("\n세그먼트별 크기:")
print(segment_size)
print("\n세그먼트별 baseline 재방문율 (is_buyer):")
print(baseline_by_segment)

reference_date 이전 평가 대상: 13529831


status
churned     11130180
retained     2399651
Name: count, dtype: int64

위험 세그먼트 크기: 2200340
status
churned     1629330
retained     571010
Name: count, dtype: int64



baseline 재방문율 (전체): 25.95%

세그먼트별 크기:
is_buyer
False    1845385
True      354955
dtype: int64

세그먼트별 baseline 재방문율 (is_buyer):
is_buyer
False    0.217863
True     0.476027
Name: status, dtype: float64


## 2. 결과 — 세그먼트 규모 및 baseline 재방문율

| 구분 | 규모 | baseline 재방문율 |
|---|---|---|
| 위험 세그먼트 전체 | 220만 명 | 25.95% |
| 세그먼트 A (고가치, `is_buyer`=True) | 35.5만 명 | 47.60% |
| 세그먼트 B (저가치, `is_buyer`=False) | 184.5만 명 | 21.79% |

- 기준 시점을 3/31로 재정의하면서 위험 세그먼트 규모가 140만 → 220만 명으로 바뀌었다
  (재정의 전 값은 4/30 기준 순환논리에 있던 잘못된 계산이므로 폐기).
- 세그먼트 A/B는 규모뿐 아니라 **baseline 재방문율 자체가 2.2배 차이** 난다 — 04에서 확인한
  "반복구매자 retained 비율이 일회성 구매자 대비 거의 2배 높다"는 패턴과 같은 맥락.
- 두 세그먼트가 캠페인 없이도 자연 회귀율이 이렇게 다른 이질적 집단이라는 점은, "하나의
  캠페인으로 실행하되 세그먼트를 층화 변수로 배정하고 결과는 세그먼트별로 나눠본다"는
  방향을 뒷받침한다 — 표본크기도 전체가 아니라 세그먼트별로 따로 계산해야 한다.

## 3. 표본크기 계산 (Power Analysis)

**가정**:
- 유의수준(α) = 0.05, 검정력(power) = 0.8 — 관례적 기본값
- MDE(최소검출효과) = **절대치 +5%p** — 두 세그먼트 모두 "재방문율을 5%p 올리면 의미있다"는
  동일한 절대 기준. baseline이 다른 두 세그먼트에 상대(%) lift 대신 절대(%p) 기준을 쓴 이유는
  해석이 더 직관적이고("재방문율이 5%p 올랐다") 업계에서도 흔히 쓰는 방식이기 때문
- 검정 방법: 두 비율 z-검정(two-proportion z-test), `statsmodels.stats.power.NormalIndPower`로
  필요 표본수를 역산

In [3]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

alpha = 0.05
power = 0.8
mde = 0.05  # 절대치 +5%p

analysis = NormalIndPower()

sample_size_results = []
for name, baseline in [
    ("전체", baseline_overall),
    ("세그먼트 A (고가치)", baseline_by_segment[True]),
    ("세그먼트 B (저가치)", baseline_by_segment[False]),
]:
    treated = baseline + mde
    effect_size = proportion_effectsize(treated, baseline)
    n_per_group = analysis.solve_power(
        effect_size=effect_size, alpha=alpha, power=power, ratio=1.0, alternative="two-sided"
    )
    sample_size_results.append({
        "segment": name,
        "baseline": baseline,
        "target": treated,
        "n_per_group": int(-(-n_per_group // 1)),  # 올림
    })

sample_size_df = pd.DataFrame(sample_size_results)
sample_size_df["n_total"] = sample_size_df["n_per_group"] * 2
sample_size_df

,segment,baseline,target,n_per_group,n_total
0,전체,0.259510,0.309510,1276,2552
1,세그먼트 A (고가치),0.476027,0.526027,1569,3138
2,세그먼트 B (저가치),0.217863,0.267863,1152,2304


## 4. 최종 설계 요약

| 구분 | baseline | 목표(+5%p) | 그룹당 필요 표본 | 실제 세그먼트 크기 |
|---|---|---|---|---|
| 전체 | 25.95% | 30.95% | 1,276명 | 220만 명 |
| 세그먼트 A (고가치) | 47.60% | 52.60% | 1,568명 | 35.5만 명 |
| 세그먼트 B (저가치) | 21.79% | 26.79% | 1,152명 | 184.5만 명 |

필요 표본(그룹당 최대 1,568명)이 실제 세그먼트 크기(최소 35.5만 명)에 비해 훨씬 작다 —
**세그먼트 A/B 둘 다 개별적으로도 검정력이 충분**하다는 뜻. 따라서 앞서 정한 방향
("하나의 캠페인으로 실행 + 세그먼트를 층화 변수로 배정 + 결과는 세그먼트별로도 분석")이
통계적으로도 문제없이 성립한다.

**대상**: 위험 세그먼트(reference_date 기준 recency 15~29일) 220만 명 전체를 캠페인
대상 풀로 삼되, `is_buyer` 기준 세그먼트 A/B로 나눠 **각 세그먼트 내에서** 실험군/대조군을
랜덤 배정(층화 무작위배정). 실제 캠페인 발송은 필요 표본보다 훨씬 큰 규모이므로, 전체
220만 명을 다 쓸 필요 없이 세그먼트별로 필요 표본의 여유분(예: 그룹당 1만 명씩, 총 4만 명)만
무작위 추출해 실행해도 충분하다.

**지표**: 04에서 확정한 이탈 정의와 동일하게, 캠페인 시작일로부터 **30일 관찰 창** 내
재방문 여부(이진 지표)

**검정 방법**: 두 비율 z-검정(two-proportion z-test, `statsmodels.stats.proportion.proportions_ztest`).
1차로 전체 실험군 vs 대조군을 비교하고, 2차로 세그먼트 A/B 각각에 대해 동일한 검정을 반복

**한계**: 세그먼트별로 검정을 두 번(A, B) 더 수행하는 것은 다중비교 문제(multiple comparisons)를
유발할 수 있다 — 우연히 유의미한 결과가 나올 확률이 커짐. 실무라면 세그먼트별 결과는
"확정적 결론"이 아니라 "탐색적 참고"로 취급하거나, 엄격하게 가려면 Bonferroni 보정
(α를 0.05/2 = 0.025로 낮추는 등)을 고려할 수 있다.